### Collection of things together now

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access AWS credentials
access_key = os.getenv("ACCESS_KEY")
secret_key = os.getenv("SECRET_KEY")

In [6]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar
import warnings
from openeo.local import LocalConnection

warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)

if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Define the STAC collection URL
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

    # Specify the spatial extent (bounding box)
    spatial_extent = {
        "west": 11.0,
        "east": 12.0,
        "south": 46.0,
        "north": 47.0
    }

    # Specify the temporal extent
    temporal_extent = ["2020-12-01", "2020-12-31"]

    # Load the data cube with specified parameters
    era5_single = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

    era5_pressure = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["t_850"]
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

    emo1 = local_conn.load_stac(
        url=stac_item,
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )

    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

    dem = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    era5_cube = era5_single.merge_cubes(era5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    remap = era5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(remap)
    print("REACHED HERE!")
    cube = remap.merge_cubes(dem_expanded)
    emo1 = emo1.rename_labels(dimension="bands", target=["target_dataset"], source=["ta24"])
    recube = cube.merge_cubes(emo1)
    
    
    # to go to raster_to_stac as UUID_X.zarr in the local

/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46291 instead
  warnings.warn(


REACHED HERE!


In [7]:
recube

In [11]:
recube.to_json()

'{\n  "process_graph": {\n    "loadstac1": {\n      "process_id": "load_stac",\n      "arguments": {\n        "spatial_extent": {\n          "west": 11.0,\n          "east": 12.0,\n          "south": 46.0,\n          "north": 47.0\n        },\n        "temporal_extent": [\n          "2020-12-01",\n          "2020-12-31"\n        ],\n        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"\n      }\n    },\n    "loadstac2": {\n      "process_id": "load_stac",\n      "arguments": {\n        "bands": [\n          "t_850"\n        ],\n        "spatial_extent": {\n          "west": 11.0,\n          "east": 12.0,\n          "south": 46.0,\n          "north": 47.0\n        },\n        "temporal_extent": [\n          "2020-12-01",\n          "2020-12-31"\n        ],\n        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"\n      }\n    },\n    "mergecubes1": {\n      "process_id": "merge_cubes",\n      "arguments": {\n        "cube1": {\n          

In [9]:
recube.execute()

/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/dask/array/core.py:1777: PerformanceWarning: Increasing number of chunks by factor of 32
  return da_func(*args, **kwargs)


<xarray.DataArray (bands: 6, time: 30, lat: 60, lon: 60)> Size: 3MB
dask.array<stack, shape=(6, 30, 60, 60), dtype=float32, chunksize=(1, 1, 5, 5), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 240B 2020-12-01 2020-12-02 ... 2020-12-30
  * lon          (lon) float64 480B 11.01 11.02 11.04 ... 11.96 11.97 11.99
  * lat          (lat) float64 480B 46.99 46.98 46.96 ... 46.04 46.03 46.01
    spatial_ref  int64 8B 0
  * bands        (bands) object 48B 'ssrd' 't2m' 'tp' ... 'dem' 'target_dataset'
Attributes: (12/22)
    GDAL:                       GDAL 3.0.4, released 2020/01/28
    GDAL_AREA_OR_POINT:         Area
    crs:                        EPSG:4326
    history_of_appended_files:  Fri Nov 20 14:22:33 2020: Appended file /huge...
    GRIB_centre:                ecmf
    GRIB_centreDescription:     European Centre for Medium-Range Weather Fore...
    ...                         ...
    history_ta24:               Sun Mar 30 07:38:14 2025: cdo -z zip_6 -P 24 ...
    keywords:                   Lisflood, Europe
    preprocessing_steps:        Assigned CRS as EPSG:4326 and clipped the dat...
    reference:                  A European daily high-resolution gridded mete...
    source:                     Lisflood Europe meteo maps - pb2015
    title:                      Lisflood meteo maps 1990-2021 for European se...

In [4]:
test1 = cube.execute()
test2 = emo1.execute()
print(set(test1.lon.values) == set(test2.lon.values))
print(test1.lon.values == test2.lon.values)

/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/dask/array/core.py:1777: PerformanceWarning: Increasing number of chunks by factor of 32
  return da_func(*args, **kwargs)


False
[ True  True  True  True False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False]


In [5]:
print(test1.lon.values, test2.lon.values)

[11.00833333 11.025      11.04166667 11.05833333 11.075      11.09166667
 11.10833333 11.125      11.14166667 11.15833333 11.175      11.19166667
 11.20833333 11.225      11.24166667 11.25833333 11.275      11.29166667
 11.30833333 11.325      11.34166667 11.35833333 11.375      11.39166667
 11.40833333 11.425      11.44166667 11.45833333 11.475      11.49166667
 11.50833333 11.525      11.54166667 11.55833333 11.575      11.59166667
 11.60833333 11.625      11.64166667 11.65833333 11.675      11.69166667
 11.70833333 11.725      11.74166667 11.75833333 11.775      11.79166667
 11.80833333 11.825      11.84166667 11.85833333 11.875      11.89166667
 11.90833333 11.925      11.94166667 11.95833333 11.975      11.99166667] [11.00833333 11.025      11.04166667 11.05833333 11.075      11.09166667
 11.10833333 11.125      11.14166667 11.15833333 11.175      11.19166667
 11.20833333 11.225      11.24166667 11.25833333 11.275      11.29166667
 11.30833333 11.325      11.34166667 11.35833333 1

In [6]:
-

SyntaxError: invalid syntax (476313318.py, line 1)

In [ ]:
test2

In [ ]:
dem.execute()

In [ ]:
emo1_change = emo1_dem.rename_labels(dimension="bands", target={"target"}, source={"ta24"})
#recube = cube.merge_cubes(emo1_change)
emo1_change

In [ ]:
recube.execute()

In [ ]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

emo1 = local_conn.load_stac(
        url=stac_item,
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
emo1.execute()


In [ ]:
emo1.execute()

In [ ]:
recube = emo1.merge_cubes(cube)
recube

In [ ]:
recube.execute()

In [ ]:
print(cube)

In [ ]:
cube._repr_html_()

In [ ]:
cube

In [ ]:
cube.to_json()

In [ ]:
cube = cube.execute()
cube = cube.to_dataset(dim="bands")
cube

In [ ]:
-

In [ ]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar
import warnings
from openeo.local import LocalConnection
import yaml

warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)

def load_config(config_path='config.yaml'):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def ensure_output_dir(output_dir):
    """Ensure output directory exists."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    return output_dir

if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)

    config = load_config()

    # Get spatial and temporal extents from config
    spatial_extent = config.get('spatial_extent', {
        "west": 6.0,
        "east": 12.0,
        "south": 37.0,
        "north": 47.0
    })
    
    temporal_extent = config.get('temporal_extent', ["2005-01-01", "2020-12-31"])

    target_variable = config.get('target_variable', "ta24")

    output_dir = ensure_output_dir(config.get('output_directory', '/data'))
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Define the STAC collection URL
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

    # Load the data cube with specified parameters
    era5_single = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

    era5_pressure = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["t_850"]
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

    emo1 = local_conn.load_stac(
        url=stac_item,
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )

    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

    dem = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    era5_cube = era5_single.merge_cubes(era5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    remap = era5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(remap)
    print("REACHED HERE!")
    cube = remap.merge_cubes(dem_expanded).execute()
    cube = cube.to_dataset(dim="bands")
    cube# to go to raster_to_stac as UUID_X.zarr in the local

In [ ]:
emo1_renamed = rename_labels(data=emo1, dimension = "bands", target = ["t2m","tp","ssrd"], source = ["ta24","pr","rg"])
y = emo1_resampled.to_dataset(dim="bands")
y = y[""] # target variable
y # To become a local stac item UUID


If incase SEAS5 is mentioned! be it a list of files or SEAS5 in general - Loop the same shit you do for ERA5

In [ ]:
import xarray as xr

ds = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/PAPER/v1/ERA5_BASE/ERA5_BASE_DAILY_2000_2020_T2M_SSRD_TP.zarr")
ds

In [ ]:
ds = ds.sel(time=slice("2000-01-01", "2000-01-31"))

In [ ]:
import xarray as xr
from datetime import datetime, timezone
from raster2stac import Raster2STAC
import logging
import os
import numpy as np

EURAC_RESEARCH_PROVIDER = {
                            "name": "Eurac Research - Institute for Earth Observation",
                            "url": "http://www.eurac.edu",
                            "roles": [
                            "processor"
                            ],
                        }
UUID = "12345"

rs2stac = Raster2STAC(
    data = ds, # Using only two variables to speed up the test
    write_collection_assets=True,
    collection_id = f"{UUID}_test_local_zarr", # The Collection id we want to set
    description = "EMO1 dataset for daily temperature, precipitation, solar radiation and potential evapo-transpiration calculated from daily temperature and solar radiation using Jensen Haise method for the years 2000 to 2022",
    license = "Apache-2.0",
    keywords = ["interTwin","EMO1","Zarr"],
    collection_url = "https://stac.intertwin.fedcloud.eu/collections/", # The URL where the collection will be 
    output_folder=f"/home/sdhinakaran/consoldiated_downScaleML/openEO-downScaleML/notebooks/{UUID}_",
    providers=[EURAC_RESEARCH_PROVIDER],
    sci_citation="Gomes, Goncalo; Thiemig, Vera; Skøien, Jon Olav; Ziese, Markus; Rauthe-Schöch, Armin; Rustemeier, Elke; Rehfeldt, Kira; Walawender, Jakub; Kolbe, Christine; Pichon, Damien; Schweim, Christoph; Salamon, Peter (2020): EMO: A high-resolution multi-variable gridded meteorological data set for Europe. European Commission, Joint Research Centre (JRC) [Dataset] doi: 10.2905/0BD84BE4-CEC8-4180-97A6-8B3ADAAC4D26 PID: http://data.europa.eu/89h/0bd84be4-cec8-4180-97a6-8b3adaac4d26",
    sci_doi="10.2905/0BD84BE4-CEC8-4180-97A6-8b3adaac4d26",
    links=[{"rel": "cite-as","href": "https://data.jrc.ec.europa.eu/dataset/0bd84be4-cec8-4180-97a6-8b3adaac4d26"}],
    s3_upload=False,
).generate_zarr_stac(item_id="test_local_zarr")



In [ ]:
from ipyleaflet import Map, DrawControl
from IPython.display import display

# Create a base map
m = Map(center=(46.5, 11.5), zoom=8)

# Add drawing control (only rectangle allowed)
draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0000FF"}})
draw_control.circle = {}
draw_control.polyline = {}
draw_control.polygon = {}
draw_control.marker = {}

spatial_extent = {}

# Callback function to handle drawing
def handle_draw(target, action, geo_json):
    global spatial_extent
    coords = geo_json['geometry']['coordinates'][0]
    # Coordinates are in [lng, lat] format
    lats = [coord[1] for coord in coords]
    lngs = [coord[0] for coord in coords]
    
    spatial_extent = {
        "west": min(lngs),
        "east": max(lngs),
        "south": min(lats),
        "north": max(lats)
    }
    
    print("Spatial extent set to:")
    print(spatial_extent)

draw_control.on_draw(handle_draw)
m.add_control(draw_control)

display(m)


In [ ]:
spatial_extent

In [ ]:
from ipyleaflet import Map, DrawControl, Rectangle
from ipywidgets import Output, HTML
from IPython.display import display
import json

# Create a map centered somewhere (e.g., Europe)
m = Map(center=(46.5, 11.5), zoom=6)

# Create output widget to capture the drawn rectangle
output = Output()
display(m, output)

# Create a draw control
dc = DrawControl()

# Only allow rectangle drawing
dc.rectangle = {
    "shapeOptions": {
        "fillColor": "#fca45d",
        "color": "#fca45d",
        "fillOpacity": 0.5
    }
}
dc.polyline = {}
dc.polygon = {}
dc.circle = {}
dc.circlemarker = {}

# Add the draw control to the map
m.add_control(dc)

# Initialize the spatial extent dictionary
spatial_extent = {}

# Function to handle draw events
def handle_draw(target, action, geo_json):
    with output:
        if action == 'created':
            # Get the bounds of the drawn rectangle
            coordinates = geo_json['geometry']['coordinates'][0]
            lons = [coord[0] for coord in coordinates]
            lats = [coord[1] for coord in coordinates]
            
            # Update the spatial_extent dictionary
            spatial_extent.update({
                "west": min(lons),
                "east": max(lons),
                "south": min(lats),
                "north": max(lats)
            })
            
            print("Spatial extent updated:")
            print(json.dumps(spatial_extent, indent=4))
            
            # You can now use spatial_extent in your code
            # For example: m.add_layer(Rectangle(bounds=((spatial_extent['south'], spatial_extent['west']), 
            #                                      (spatial_extent['north'], spatial_extent['east']))))

# Register the callback
dc.on_draw(handle_draw)

# Instructions for the user
html = HTML('''<b>Instructions:</b> Draw a rectangle on the map to define your spatial extent.
             <br>The coordinates will be automatically stored in the <code>spatial_extent</code> variable.''')
display(html)

In [ ]:
from ipywidgets import widgets, Layout
from datetime import date
from IPython.display import display

# Initialize temporal_extent
temporal_extent = []

# Create date pickers with reasonable defaults
start_date_picker = widgets.DatePicker(
    description='Start Date:',
    value=date(2000, 1, 1),
    layout=Layout(width='300px')
)

end_date_picker = widgets.DatePicker(
    description='End Date:',
    value=date(2020, 12, 31),
    layout=Layout(width='300px')
)

# Create output widget to display the selected range
output = widgets.Output()

# Button to confirm selection
button = widgets.Button(description="Set Temporal Extent")

def on_button_click(b):
    with output:
        temporal_extent.clear()
        temporal_extent.extend([
            start_date_picker.value.strftime("%Y-%m-%d"),
            end_date_picker.value.strftime("%Y-%m-%d")
        ])
        print(f"Training Period extent set to: {temporal_extent}")

button.on_click(on_button_click)

# Display the widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Training Period</h3>"),
    start_date_picker,
    end_date_picker,
    button,
    output
]))

In [ ]:
from ipywidgets import widgets, Layout
from IPython.display import display

# Initialize target_variable
target_variable = None

# Create radio buttons
radio = widgets.RadioButtons(
    options=["t2m", "ssrd", "tp"],
    description='Target Variable:',
    disabled=False,
    layout=Layout(width='200px')
)

# Create output widget
output = widgets.Output()

def on_change(change):
    global target_variable
    if change['type'] == 'change' and change['name'] == 'value':
        target_variable = change['new']
        with output:
            output.clear_output()
            print(f"Selected target variable: {target_variable}")

radio.observe(on_change)

# Display the widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Target Variable</h3>"),
    radio,
    output
]))

## Experimenting the boxes

In [ ]:
import json
from openeo_pg_parser_networkx import OpenEOProcessGraph
import matplotlib.pyplot as plt

# The process graph JSON
process_graph_json = {
  "process_graph": {
    "loadstac1": {
      "process_id": "load_stac",
      "arguments": {
        "spatial_extent": {
          "west": 11.0,
          "east": 12.0,
          "south": 46.0,
          "north": 47.0
        },
        "temporal_extent": [
          "2018-01-01",
          "2020-12-31"
        ],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
      }
    },
    "loadstac2": {
      "process_id": "load_stac",
      "arguments": {
        "bands": [
          "t_850"
        ],
        "spatial_extent": {
          "west": 11.0,
          "east": 12.0,
          "south": 46.0,
          "north": 47.0
        },
        "temporal_extent": [
          "2018-01-01",
          "2020-12-31"
        ],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
      }
    },
    "mergecubes1": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {
          "from_node": "loadstac1"
        },
        "cube2": {
          "from_node": "loadstac2"
        }
      }
    },
    "loadstac3": {
      "process_id": "load_stac",
      "arguments": {
        "bands": [
          "dem"
        ],
        "spatial_extent": {
          "west": 11.0,
          "east": 12.0,
          "south": 46.0,
          "north": 47.0
        },
        "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
      }
    },
    "resamplecubespatial1": {
      "process_id": "resample_cube_spatial",
      "arguments": {
        "data": {
          "from_node": "mergecubes1"
        },
        "method": "bilinear",
        "target": {
          "from_node": "loadstac3"
        }
      }
    },
    "resamplecubetemporal1": {
      "process_id": "resample_cube_temporal",
      "arguments": {
        "data": {
          "from_node": "loadstac3"
        },
        "target": {
          "from_node": "resamplecubespatial1"
        }
      }
    },
    "mergecubes2": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {
          "from_node": "resamplecubespatial1"
        },
        "cube2": {
          "from_node": "resamplecubetemporal1"
        }
      },
      "result": True
    }
  }
}

# Use the process graph from memory (not from file)
parsed_graph = OpenEOProcessGraph.from_dict(process_graph_json["process_graph"])

# Plot with interactive UI (zoom/pan/hover/click)
parsed_graph.plot(interactive=True)

In [ ]:
import json

process_graph_json = {
  "process_graph": {
    "loadstac1": {
      "process_id": "load_stac",
      "arguments": {
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
      }
    },
    "loadstac2": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["t_850"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
      }
    },
    "mergecubes1": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "loadstac1"},
        "cube2": {"from_node": "loadstac2"}
      }
    },
    "loadstac3": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["dem"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
      }
    },
    "resamplecubespatial1": {
      "process_id": "resample_cube_spatial",
      "arguments": {
        "data": {"from_node": "mergecubes1"},
        "method": "bilinear",
        "target": {"from_node": "loadstac3"}
      }
    },
    "resamplecubetemporal1": {
      "process_id": "resample_cube_temporal",
      "arguments": {
        "data": {"from_node": "loadstac3"},
        "target": {"from_node": "resamplecubespatial1"}
      }
    },
    "mergecubes2": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "resamplecubespatial1"},
        "cube2": {"from_node": "resamplecubetemporal1"}
      },
      "result": True
    }
  }
}

# Save to file
with open("process_graph.json", "w") as f:
    json.dump(process_graph_json["process_graph"], f, indent=2)


In [ ]:
from openeo_pg_parser_networkx import OpenEOProcessGraph

# Load the process graph from the file
parsed_graph = OpenEOProcessGraph.from_file("process_graph.json")

# Show interactive graph
parsed_graph.plot()


In [ ]:
cube

In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
from openeo_pg_parser_networkx import ProcessGraphParser
from dask.distributed import Client
import dask
from dask import visualize

# Your JSON process graph
process_graph_json = {
  "process_graph": {
    "loadstac1": {
      "process_id": "load_stac",
      "arguments": {
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
      }
    },
    "loadstac2": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["t_850"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
      }
    },
    "mergecubes1": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "loadstac1"},
        "cube2": {"from_node": "loadstac2"}
      }
    },
    "loadstac3": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["dem"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
      }
    },
    "resamplecubespatial1": {
      "process_id": "resample_cube_spatial",
      "arguments": {
        "data": {"from_node": "mergecubes1"},
        "method": "bilinear",
        "target": {"from_node": "loadstac3"}
      }
    },
    "resamplecubetemporal1": {
      "process_id": "resample_cube_temporal",
      "arguments": {
        "data": {"from_node": "loadstac3"},
        "target": {"from_node": "resamplecubespatial1"}
      }
    },
    "mergecubes2": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "resamplecubespatial1"},
        "cube2": {"from_node": "resamplecubetemporal1"}
      },
      "result": True
    }
  }
}

# 1. Parse the process graph into a networkx graph
parser = ProcessGraphParser()
G = parser.parse_to_networkx(process_graph_json["process_graph"])

# 2. Visualize the networkx graph (static)
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)  # Layout for better node positioning
nx.draw(G, pos, with_labels=True, node_size=2000, node_color="skyblue", font_size=10, arrowsize=20)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): d["process_id"] for u, v, d in G.edges(data=True)}, font_size=8)
plt.title("openEO Process Graph Visualization (networkx)")
plt.tight_layout()
plt.show()

# 3. Create a Dask task graph (mock execution)
# Note: This requires actual openEO backend integration for real execution
# Here we'll create a mock Dask graph for visualization purposes

def mock_process(node_id, process_id, **kwargs):
    """Mock function representing an openEO process"""
    return dask.delayed(lambda x: x)(node_id)

# Build a mock Dask graph
mock_graph = {}
for node_id, node_data in process_graph_json["process_graph"].items():
    process_id = node_data["process_id"]
    args = node_data.get("arguments", {})
    
    # Handle dependencies (from_node references)
    dependencies = []
    for arg_name, arg_value in args.items():
        if isinstance(arg_value, dict) and "from_node" in arg_value:
            dependencies.append(mock_graph[arg_value["from_node"]])
    
    mock_graph[node_id] = mock_process(node_id, process_id, **args)

# Get the final result node
result_node = next(n for n, d in process_graph_json["process_graph"].items() if d.get("result", False))
dask_graph = mock_graph[result_node]

# 4. Visualize the Dask task graph (interactive)
# Requires graphviz installed: `conda install python-graphviz` or `pip install graphviz`
dask.visualize(dask_graph, filename="dask_graph.svg", engine="cytoscape", 
               node_attr={"style": "filled", "fillcolor": "lightblue"})
print("Dask task graph saved as 'dask_graph.svg' (open in browser for interactivity)")

In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
from openeo_pg_parser_networkx import OpenEOProcessGraph
from dask.distributed import Client
import dask
from dask import visualize

# Your JSON process graph
process_graph_json = {
  "process_graph": {
    "loadstac1": {
      "process_id": "load_stac",
      "arguments": {
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
      }
    },
    "loadstac2": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["t_850"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "temporal_extent": ["2018-01-01", "2020-12-31"],
        "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
      }
    },
    "mergecubes1": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "loadstac1"},
        "cube2": {"from_node": "loadstac2"}
      }
    },
    "loadstac3": {
      "process_id": "load_stac",
      "arguments": {
        "bands": ["dem"],
        "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
        "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
      }
    },
    "resamplecubespatial1": {
      "process_id": "resample_cube_spatial",
      "arguments": {
        "data": {"from_node": "mergecubes1"},
        "method": "bilinear",
        "target": {"from_node": "loadstac3"}
      }
    },
    "resamplecubetemporal1": {
      "process_id": "resample_cube_temporal",
      "arguments": {
        "data": {"from_node": "loadstac3"},
        "target": {"from_node": "resamplecubespatial1"}
      }
    },
    "mergecubes2": {
      "process_id": "merge_cubes",
      "arguments": {
        "cube1": {"from_node": "resamplecubespatial1"},
        "cube2": {"from_node": "resamplecubetemporal1"}
      },
      "result": True
    }
  }
}

# 1. Parse the process graph into a networkx graph
# Correct way to initialize OpenEOProcessGraph
process_graph = OpenEOProcessGraph(pg_data=process_graph_json["process_graph"])
G = process_graph.to_networkx()

# 2. Visualize the networkx graph (static)
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=2000, node_color="skyblue", font_size=10, arrowsize=20)

# Improved edge labels showing the relationship
edge_labels = {
    (u, v): f"{G.nodes[u]['process_id']} → {G.nodes[v]['process_id']}"
    for u, v in G.edges()
}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title("openEO Process Graph Visualization (networkx)")
plt.tight_layout()
plt.show()

# 3. Create a Dask task graph (mock execution)
def mock_process(node_id, process_id, **kwargs):
    """Mock function representing an openEO process"""
    return dask.delayed(lambda x: x)(node_id)

mock_graph = {}
for node_id, node_data in process_graph_json["process_graph"].items():
    process_id = node_data["process_id"]
    args = node_data.get("arguments", {})
    
    # Handle dependencies
    dependencies = []
    for arg_value in args.values():
        if isinstance(arg_value, dict) and "from_node" in arg_value:
            dependencies.append(mock_graph[arg_value["from_node"]])
    
    # Create mock process with dependencies
    if dependencies:
        mock_graph[node_id] = mock_process(node_id, process_id, **args).visualize()
    else:
        mock_graph[node_id] = mock_process(node_id, process_id, **args)

# Get the final result node
result_node = next(n for n, d in process_graph_json["process_graph"].items() 
                  if d.get("result", False))
dask_graph = mock_graph[result_node]

# 4. Visualize the Dask task graph
try:
    dask.visualize(
        dask_graph, 
        filename="dask_graph.svg",
        engine="cytoscape",
        node_attr={"style": "filled", "fillcolor": "lightblue"}
    )
    print("Dask task graph saved as 'dask_graph.svg'")
except Exception as e:
    print(f"Error visualizing Dask graph: {e}")
    print("Make sure you have graphviz installed: pip install graphviz")

In [ ]:
# Get the current process graph as a Python dict
process_graph = cube.flat_graph()

# Modify the graph (e.g., change bands or spatial extent)
process_graph["loadstac2"]["arguments"]["bands"] = ["t_1000"]  # Fake band name
process_graph["loadstac1"]["arguments"]["temporal_extent"] = ["2019-01-01", "2020-12-31"]

# Rebuild the cube with the modified graph
from openeo.rest.datacube import DataCube
modified_cube = DataCube.load_json(process_graph)

# Trigger the widget with fake data
modified_cube  # Jupyter will now render the modified version

In [ ]:
from openeo.rest.datacube import DataCube

# Get the current process graph
process_graph = cube.flat_graph()

# Modify the graph (e.g., change bands or extent)
process_graph["loadstac2"]["arguments"]["bands"] = ["t_1000"]  # Fake band
process_graph["loadstac1"]["arguments"]["temporal_extent"] = ["2019-01-01", "2020-12-31"]

from openeo.rest.datacube import DataCube

# Manually inject the process graph into a new DataCube instance
modified_cube = DataCube(pg_node=process_graph, connection=cube.connection)
modified_cube  # May still fail for local connections

In [ ]:
dir(cube)

In [ ]:
cube._pg

In [ ]:
# Get the process graph
graph = cube.flat_graph()

# Inject a "fake" band math operation
graph["custom_band"] = {
    "process_id": "array_element",
    "arguments": {"data": {"from_node": "loadstac1"}, "index": 0}
}

# Rebuild the cube (remote backends only)
modified_cube = cube.connection.datacube_from_process_graph(graph)
modified_cube  # Renders with your injected band

In [11]:
import xarray as xr

ds = xr.open_dataset("/home/sdhinakaran/test/seasonal_fc_20020701.nc")
ds

<xarray.Dataset> Size: 2GB
Dimensions:    (longitude: 73, latitude: 49, number: 25, time: 861)
Coordinates:
  * longitude  (longitude) float32 292B 2.0 2.25 2.5 2.75 ... 19.5 19.75 20.0
  * latitude   (latitude) float32 196B 52.0 51.75 51.5 51.25 ... 40.5 40.25 40.0
  * number     (number) int32 100B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time       (time) datetime64[ns] 7kB 2002-07-01 ... 2003-02-01
Data variables:
    t2m        (time, number, latitude, longitude) float64 616MB ...
    ssrd       (time, number, latitude, longitude) float64 616MB ...
    tp         (time, number, latitude, longitude) float64 616MB ...
Attributes:
    Conventions:  CF-1.6
    history:      2025-06-23 17:39:44 GMT by grib_to_netcdf-2.39.4: /opt/ecmw...

In [2]:
import matplotlib.pyplot as plt
import networkx as nx
from openeo_pg_parser_networkx import OpenEOProcessGraph

pg = OpenEOProcessGraph.from_file("jsonn.json")
G = pg.to_networkx()

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=2000, font_size=10)
plt.title("OpenEO Process Graph")
plt.show()


AttributeError: 'OpenEOProcessGraph' object has no attribute 'to_networkx'

In [1]:
from IPython.display import HTML

# Your openEO process graph JSON
process_graph_json = """
{
  "id": "23ed23dd5f8a4663a3b7a803fd38292d",
  "explicit-zoom": true,
  "height": "400px",
  "value": {
    "process_graph": {
      "loadstac1": {
        "process_id": "load_stac",
        "arguments": {
          "spatial_extent": {
            "west": 11.0,
            "east": 12.0,
            "south": 46.0,
            "north": 47.0
          },
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
        }
      },
      "loadstac2": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["t_850"],
          "spatial_extent": {
            "west": 11.0,
            "east": 12.0,
            "south": 46.0,
            "north": 47.0
          },
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
        }
      },
      "mergecubes1": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {
            "from_node": "loadstac1"
          },
          "cube2": {
            "from_node": "loadstac2"
          }
        }
      },
      "loadstac3": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["dem"],
          "spatial_extent": {
            "west": 11.0,
            "east": 12.0,
            "south": 46.0,
            "north": 47.0
          },
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
        }
      },
      "resamplecubespatial1": {
        "process_id": "resample_cube_spatial",
        "arguments": {
          "data": {
            "from_node": "mergecubes1"
          },
          "method": "bilinear",
          "target": {
            "from_node": "loadstac3"
          }
        }
      },
      "resamplecubetemporal1": {
        "process_id": "resample_cube_temporal",
        "arguments": {
          "data": {
            "from_node": "loadstac3"
          },
          "target": {
            "from_node": "resamplecubespatial1"
          }
        }
      },
      "mergecubes2": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {
            "from_node": "resamplecubespatial1"
          },
          "cube2": {
            "from_node": "resamplecubetemporal1"
          }
        }
      },
      "loadstac4": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["ta24"],
          "spatial_extent": {
            "west": 11.0,
            "east": 12.0,
            "south": 46.0,
            "north": 47.0
          },
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"
        }
      },
      "renamelabels1": {
        "process_id": "rename_labels",
        "arguments": {
          "data": {
            "from_node": "loadstac4"
          },
          "dimension": "bands",
          "source": ["ta24"],
          "target": ["target_dataset"]
        }
      },
      "mergecubes3": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {
            "from_node": "mergecubes2"
          },
          "cube2": {
            "from_node": "renamelabels1"
          }
        },
        "result": true
      }
    }
  }
}
"""

# HTML template to load the Model Builder
html_template = f"""
<div id="openeo-model-builder-container">
    <script>
    if (!window.customElements || !window.customElements.get('openeo-model-builder')) {{
        var el = document.createElement('script');
        el.src = "https://cdn.jsdelivr.net/npm/@openeo/vue-components@2/assets/openeo.min.js";
        document.head.appendChild(el);

        var font = document.createElement('font');
        font.as = "font";
        font.type = "font/woff2";
        font.crossOrigin = true;
        font.href = "https://use.fontawesome.com/releases/v5.13.0/webfonts/fa-solid-900.woff2";
        document.head.appendChild(font);
    }}
    </script>
    <openeo-model-builder>
        <script type="application/json">
            {process_graph_json}
        </script>
    </openeo-model-builder>
</div>
"""

# Display in Jupyter Notebook
HTML(html_template)

In [3]:
from IPython.display import HTML

# Your openEO process graph JSON
process_graph_json = """
{
  "id": "23ed23dd5f8a4663a3b7a803fd38292d",
  "explicit-zoom": true,
  "height": "400px",
  "value": {
    "process_graph": {
      "loadstac1": {
        "process_id": "load_stac",
        "arguments": {
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
        }
      },
      "loadstac2": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["t_850"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
        }
      },
      "mergecubes1": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "loadstac1"},
          "cube2": {"from_node": "loadstac2"}
        }
      },
      "loadstac3": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["dem"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
        }
      },
      "resamplecubespatial1": {
        "process_id": "resample_cube_spatial",
        "arguments": {
          "data": {"from_node": "mergecubes1"},
          "method": "bilinear",
          "target": {"from_node": "loadstac3"}
        }
      },
      "resamplecubetemporal1": {
        "process_id": "resample_cube_temporal",
        "arguments": {
          "data": {"from_node": "loadstac3"},
          "target": {"from_node": "resamplecubespatial1"}
        }
      },
      "mergecubes2": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "resamplecubespatial1"},
          "cube2": {"from_node": "resamplecubetemporal1"}
        }
      },
      "loadstac4": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["ta24"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"
        }
      },
      "renamelabels1": {
        "process_id": "rename_labels",
        "arguments": {
          "data": {"from_node": "loadstac4"},
          "dimension": "bands",
          "source": ["ta24"],
          "target": ["target_dataset"]
        }
      },
      "mergecubes3": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "mergecubes2"},
          "cube2": {"from_node": "renamelabels1"}
        }
      },
      "runoscar1": {
        "process_id": "run_oscar",
        "arguments": {
          "data": {"from_node": "mergecubes3"},  // <-- Connect mergecubes3 as input
          "local_refresh_token": "/home/jzvolensky/.local/share/openeo-python-client/refresh-tokens.json",
          "oscar_endpoint": "https://oscar-grnet.intertwin.fedcloud.eu",
          "output": "output",
          "service": "hydroform-hydromt",
          "service_config": "./configs/oscar_hydromt_svc.yaml",
          "token_env_var": "$TOKEN"
        },
        "result": true  // <-- Final output
      }
    }
  }
}
"""

# HTML template (same as before)
html_template = f"""
<div id="openeo-model-builder-container">
    <script>
    if (!window.customElements || !window.customElements.get('openeo-model-builder')) {{
        var el = document.createElement('script');
        el.src = "https://cdn.jsdelivr.net/npm/@openeo/vue-components@2/assets/openeo.min.js";
        document.head.appendChild(el);

        var font = document.createElement('font');
        font.as = "font";
        font.type = "font/woff2";
        font.crossOrigin = true;
        font.href = "https://use.fontawesome.com/releases/v5.13.0/webfonts/fa-solid-900.woff2";
        document.head.appendChild(font);
    }}
    </script>
    <openeo-model-builder>
        <script type="application/json">
            {process_graph_json}
        </script>
    </openeo-model-builder>
</div>
"""

# Display in Jupyter
HTML(html_template)

In [14]:
from IPython.display import HTML

process_graph_json = """
{
  "id": "23ed23dd5f8a4663a3b7a803fd38292d",
  "explicit-zoom": true,
  "height": "400px",
  "value": {
    "process_graph": {
      "loadstac1": {
        "process_id": "load_stac",
        "arguments": {
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"
        }
      },
      "loadstac2": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["t_850"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"
        }
      },
      "mergecubes1": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "loadstac1"},
          "cube2": {"from_node": "loadstac2"}
        }
      },
      "loadstac3": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["dem"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"
        }
      },
      "resamplecubespatial1": {
        "process_id": "resample_cube_spatial",
        "arguments": {
          "data": {"from_node": "mergecubes1"},
          "method": "bilinear",
          "target": {"from_node": "loadstac3"}
        }
      },
      "resamplecubetemporal1": {
        "process_id": "resample_cube_temporal",
        "arguments": {
          "data": {"from_node": "loadstac3"},
          "target": {"from_node": "resamplecubespatial1"}
        }
      },
      "mergecubes2": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "resamplecubespatial1"},
          "cube2": {"from_node": "resamplecubetemporal1"}
        }
      },
      "loadstac4": {
        "process_id": "load_stac",
        "arguments": {
          "bands": ["ta24"],
          "spatial_extent": {"west": 11.0, "east": 12.0, "south": 46.0, "north": 47.0},
          "temporal_extent": ["2020-12-01", "2020-12-31"],
          "url": "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"
        }
      },
      "renamelabels1": {
        "process_id": "rename_labels",
        "arguments": {
          "data": {"from_node": "loadstac4"},
          "dimension": "bands",
          "source": ["ta24"],
          "target": ["target_dataset"]
        }
      },
      "mergecubes3": {
        "process_id": "merge_cubes",
        "arguments": {
          "cube1": {"from_node": "mergecubes2"},
          "cube2": {"from_node": "renamelabels1"}
        }
      },
      "runoscar1": {
        "process_id": "run_oscar",
        "arguments": {
          "data": {"from_node": "mergecubes3"},
          "local_refresh_token": "/home/sdhinakaran/.local/share/openeo-python-client/refresh-tokens.json",
          "oscar_endpoint": "https://oscar-grnet.intertwin.fedcloud.eu",
          "service": "downScaleML",
          "service_config": "./configs/oscar_hydromt_svc.yaml",
          "token_env_var": "$TOKEN",
          "output": "output"
        },
        "result": true
      }
    }
  }
}
"""

html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <title>openEO Process Graph</title>
    <script src="https://cdn.jsdelivr.net/npm/@openeo/vue-components@2/assets/openeo.min.js"></script>
    <link rel="stylesheet" href="https://use.fontawesome.com/releases/v5.13.0/css/all.css">
</head>
<body>
    <openeo-model-builder style="height: 600px; width: 100%;">
        <script type="application/json">
            {process_graph_json}
        </script>
    </openeo-model-builder>
</body>
</html>
"""

HTML(html_template)